In [4]:
import os
import numpy as np
from numpy.polynomial.polynomial import Polynomial
import matplotlib.pyplot as plt
import time
import socket
from pyModbusTCP.client import ModbusClient
# import RPi.GPIO as GPIO

In [5]:
urHost = "192.168.50.155"
socketPort = 30002  # The same port as used by the server, or 30003
modbusPort = 502
registers = list(range(280, 286))


def socketConnect(host, port):
    socketConn = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    socketConn.connect((host, port))
    return socketConn


def modbusConnect(host, port):
    modbusConn = ModbusClient(host=host, port=port, auto_open=True)
    return modbusConn


print("Connecting to UR and Modbus...")
urConnTime = time.time()
socketConn = socketConnect(urHost, socketPort)
modbusConn = modbusConnect(urHost, 502)
eclipseTime = time.time() - urConnTime
print("Connection time: ", eclipseTime)

# ??URP???
baseSrc = 'def myURP():\n set_tcp(p[0.0,0.0,0.0,0.0,0.0,0.0])\n set_payload(1.5)\n set_standard_analog_input_domain(0, 0)\n set_standard_analog_input_domain(1, 0)\n set_tool_analog_input_domain(0, 1)\n set_tool_analog_input_domain(1, 1)\n set_analog_outputdomain(0, 0)\n set_analog_outputdomain(1, 0)\n set_input_actions_to_default()\n set_gravity([0.0, 0.0, 9.82])\n set_safety_mode_transition_hardness(1)\n set_tool_voltage(0)\n global i_var_1=1global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n'
movelSrcHead = "cur_pt=get_actual_tcp_pose()\n"
movelSrcPoint = "z_move=p[0,{},0,0,0,0]\n"
movelSrcTail = "new_pt=pose_add(cur_pt,z_move)\nmovel(new_pt,a=1.3962634015954636, v=1.0471975511965976)\n"
# movelSrcTail = "new_pt=pose_add(cur_pt,z_move)\nmovel(new_pt,a=1.0962634015954636, v=1.0471975511965976)\n"
RG2Src = "RG2(100,40.0,True,False,False)\nend\n"

Connecting to UR and Modbus...
Connection time:  0.0010001659393310547


In [6]:
def wait_UR_state():
    # managing TCP sessions with call to c.open()/c.close()
    modbusConn.open()
    # print("modbus")

    jump_point = 0
    while jump_point == 0:
        global joint_1, joint_2, joint_3, joint_4, joint_5, joint_6
        # print("Read Joint!!")
        time.sleep(0.5)
        joint_1 = 1
        joint_2 = 1
        joint_3 = 1
        joint_4 = 1
        joint_5 = 1
        joint_6 = 1

        joint_280 = modbusConn.read_holding_registers(registers[0], 1)
        joint_281 = modbusConn.read_holding_registers(registers[1], 1)
        joint_282 = modbusConn.read_holding_registers(registers[2], 1)
        joint_283 = modbusConn.read_holding_registers(registers[3], 1)
        joint_284 = modbusConn.read_holding_registers(registers[4], 1)
        joint_285 = modbusConn.read_holding_registers(registers[5], 1)

        if joint_280:
            joint_1 = joint_280[0]
        if joint_281:
            joint_2 = joint_281[0]
        if joint_282:
            joint_3 = joint_282[0]
        if joint_283:
            joint_4 = joint_283[0]
        if joint_284:
            joint_5 = joint_284[0]
        if joint_285:
            joint_6 = joint_285[0]

        if (
            joint_1 == 0
            and joint_2 == 0
            and joint_3 == 0
            and joint_4 == 0
            and joint_5 == 0
            and joint_6 == 0
        ):
            # print("Move done!")
            jump_point = 1
            
def RGRelease():
    cmd = "RG2(100,10.0,True,False,False)\nend\n"
    strL = "{}{}{}".format(baseSrc, cmd, "end\n")
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def RGGrab():
    cmd = "RG2(10,10.0,True,False,False)\nend\n"
    strL = "{}{}{}".format(baseSrc, cmd, "end\n")
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def Point0():
    cmd = "movej(get_inverse_kin(p[.279134293245, .045957778215, .218270318258, 2.216666768333, -2.213592243070, -.007045821207], qnear=[0.5594176650047302, -1.0216768423663538, -2.0281041304217737, -1.6719406286822718, 1.5753320455551147, -2.5834994951831263]), a=1.3962634015954636, v=1.0471975511965976)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    
def Point_ball():
    cmd = "movej(get_inverse_kin(p[.646928316386, .496238388949, .074590321335, 3.115985415892, -.027590097534, .068245882756], qnear=[0.8072443008422852, -2.4104560057269495, -0.8106430212603968, -1.4788373152362269, 1.5237241983413696, -3.889764372502462]), a=1.3962634015954636, v=1.0471975511965976)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()

def Point_cube():
    cmd = "movej(get_inverse_kin(p[.530955284719, .496186040850, .074612719288, 3.115902526355, -.027555485725, .068321532283], qnear=[0.9231362342834473, -2.169014279042379, -1.2252424399005335, -1.311493221913473, 1.5225478410720825, -3.77280837694277]), a=1.3962634015954636, v=1.0471975511965976)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()

def Point_ring():
    cmd = "movej(get_inverse_kin(p[.424987346440, .496179568406, .074594930094, 3.115875260401, -.027478327548, .068438622616], qnear=[1.053391695022583, -2.000748936329977, -1.4857175985919397, -1.225849453602926, 1.5220800638198853, -3.641836229954855]), a=1.3962634015954636, v=1.0471975511965976)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()

def Point_triangle():
    cmd = "movej(get_inverse_kin(p[.290988814672, .496204446954, .074669489394, 3.116048325960, -.027622157322, .068363836976], qnear=[1.2565999031066895, -1.831412140523092, -1.7190282980548304, -1.1719391981707972, 1.5233160257339478, -3.437768046055929]), a=1.3962634015954636, v=1.0471975511965976)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()

def move_X_axis(interval):
    # ???X???????Y????
    # ?????X???????????Y???????
    interval = interval
    interval = interval / 1000
    cmd = f"{movelSrcHead}z_move=p[0,{-interval},0,0,0,0]\n{movelSrcTail}"
    strL = "{}{}{}".format(baseSrc, cmd, "end\n")
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    
def move_Y_axis(interval):
    # ????????Y???????X????
    interval = interval
    interval = interval / 1000
    cmd = f"{movelSrcHead}z_move=p[{interval},0,0,0,0,0]\n{movelSrcTail}"
    strL = "{}{}{}".format(baseSrc, cmd, "end\n")
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()


def move_Z_positive(interval):
    # Z???????????????????
    interval = interval
    interval = interval / 1000
    cmd = f"{movelSrcHead}z_move=p[0,0,{interval},0,0,0]\n{movelSrcTail}"
    strL = "{}{}{}".format(baseSrc, cmd, "end\n")
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()

def move(Xinterval, Yinterval):
    # ????????Y???????X????
    Xinterval = Xinterval / 1000
    Yinterval = Yinterval / 1000
    cmd = f"{movelSrcHead}z_move=p[{-Yinterval},{-Xinterval},0,0,0,0]\n{movelSrcTail}"
    strL = "{}{}{}".format(baseSrc, cmd, "end\n")
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()

def rotate(angle):
    # 角度轉換為弧度
    angle = angle * np.pi / 180
    cmd = f"{movelSrcHead}z_move=p[0,0,0,0,0,{angle}]\n{movelSrcTail}"
    strL = "{}{}{}".format(baseSrc, cmd, "end\n")
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()

# ???? pixel size ???
def calculate_updated_pixel_size(height):
    poly = Polynomial(
        np.array([-0.05597298, 0.06581917, -0.00017355])
    )  # ???????
    return poly(height)  # ??????? pixel size

In [21]:
RGGrab()
RGRelease()

In [15]:
import math

step = 0.02
diag = step / math.sqrt(2)

def forward():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[{step}, 0, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def backward():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[-{step}, 0, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def left():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[0, -{step}, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def right():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[0, {step}, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def left_forward():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[{diag}, -{diag}, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def right_forward():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[{diag}, {diag}, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def left_backward():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[-{diag}, -{diag}, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)

def right_backward():
    cmd = f"movel(pose_trans(get_actual_tcp_pose(), p[-{diag}, {diag}, 0, 0, 0, 0]), 1.2, 0.25)\n"
    strL = 'def myURP():\n  global measure_width=0\n  global grip_detected=False\n  global lost_grip=False\n  global zsysx=0\n  global zsysy=0\n  global zsysz=0.06935\n  global zsysm=0.7415\n  global zmasx=0\n  global zmasy=-0\n  global zmasz=0.18659\n  global zslax=0\n  global zslay=0\n  global zslaz=0\n  global zmasm=0\n  global zslam=0\n  global zslatcp=p[0,0,0,0,0,0]\n  global zmastcp=p[0,0,0.18659,0,-0,-3.14159]\n  thread lost_grip_thread():\n  while True:\n  set_tool_voltage(24)\n  \tif True ==get_digital_in(9):\n  \t\tsleep(0.024)\n  \t\tif True == grip_detected:\n  \t\t\tif False == get_digital_in(8):\n  \t\t\t\tgrip_detected=False\n  \t\t\t\tlost_grip=True\n  \t\t\tend\n  \t\tend\n  \tset_tool_analog_input_domain(0, 1)\n  \tset_tool_analog_input_domain(1, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tend\n  \tsync()\n  end\n  end\n  lg_thr = run lost_grip_thread()\n  def RG2(target_width=110, target_force=40, payload=0.0, set_payload=False, depth_compensation=False, slave=False):\n  \tset_tcp(p[0,0,0.18659,0,-0,-3.14159])\n  \tgrip_detected=False\n  \tif slave:\n  \t\tslave_grip_detected=False\n  \telse:\n  \t\tmaster_grip_detected=False\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout = timeout+1\n  \t  sync()\n  \tend\n\tdef bit(input):\n  \t  msb=65536\n  \t  local i=0\n  \t  local output=0\n  \t  while i<17:\n\t\tset_digital_out(8,True)\n  \t    if input>=msb:\n  \t      input=input-msb\n  \t      set_digital_out(9,False)\n  \t    else:\n  \t      set_digital_out(9,True)\n  \t    end\n  \t    if get_digital_in(8):\n  \t      out=1\n  \t    end\n  \t    sync()\n  \t    set_digital_out(8,False)\n  \t    sync()\n  \t    input=input*2\n  \t    output=output*2\n  \t    i=i+1\n  \t  end\n  \t  return output\n  \tend\n  \ttarget_width=target_width+9.2\n  \tif target_force>40:\n  \ttarget_force=40\n  \tend\n  \tif target_force<3:\n  \ttarget_force=3\n  \tend\n  \tif target_width>110:\n  \ttarget_width=110\n  \tend\n  \tif target_width<0:\n  \ttarget_width=0\n  \tend\n  \trg_data=floor(target_width)*4\n  \trg_data=rg_data+floor(target_force/2)*4*111\n  \trg_data=rg_data+32768\n  \tif slave:\n  \trg_data=rg_data+16384\n  \tend\n  \tbit(rg_data)\n  \tif slave==False:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zmastcp)\n  \tend\n  \tif slave:\n  \tt_w_rg=pose_trans(get_actual_tool_flange_pose(), zslatcp)\n  \tend\n  \tt_rg_w=pose_inv(t_w_rg)\n  \tif depth_compensation:\n  \tfinger_length = 55.0/1000\n  \tfinger_heigth_disp = 5.0/1000\n  \tcenter_displacement = 7.5/1000\n\n  \tstart_pose = get_forward_kin()\n  \tset_analog_inputrange(2, 1)\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n\n  \tstart_depth = cos(zangle)*finger_length\n\n  \tsleep(0.016)\n  \ttimeout = 0\n  \twhile get_digital_in(9) == True:\n  \t  timeout=timeout+1\n  \t  sleep(0.008)\n  \t  if timeout > 20:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \tcompensation_depth = 0\n  \twhile get_digital_in(9) == False:\n  \t  zscale = (get_analog_in(2)-0.026)/2.9760034\n  \t  zangle = zscale*1.57079633+-0.08726646\n  \t  zwidth = 5.0+110*sin(zangle)\n  \t  measure_depth = cos(zangle)*finger_length\n  \t  compensation_depth = (measure_depth - start_depth)\n  \t  target_pose =pose_add(start_pose,pose_trans(pose_trans(t_w_rg, p[0,0,-compensation_depth,0,0,0]),t_rg_w))\n\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \t  timeout=timeout+1\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tif point_dist(target_pose, get_forward_kin()) > 0.005:\n  \tpopup("Lower grasping force or max width",title="RG-lag threshold exceeded", warning=False, error=False, blocking=False)\n  \tend\n  \tend\n  \tact_comp_pose = p[0,0,0,0,0,0]\n  \twhile norm(act_comp_pose) < norm(compensation_depth)-0.0002:\n  \tservoj(get_inverse_kin(target_pose),0,0,0.008,0.01,2000)\n  \tact_comp_pose = pose_trans(pose_inv(start_pose),get_forward_kin())\n  \tend\n  \tstopj(2)\n  \tend\n  \tif depth_compensation==False:\n  \ttimeout = 0\n  \ttimeout_count=20*0.008/0.008\n  \twhile get_digital_in(9) == True:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_count:\n  \t    break\n  \t  end\n  \tend\n  \ttimeout = 0\n  \ttimeout_limit = 750000\n  \twhile get_digital_in(9) == False:\n  \t  timeout = timeout+1\n  \t  sync()\n  \t  if timeout > timeout_limit:\n  \t    break\n  \t  end\n  \tend\n  \tend\n  \tsleep(0.024)\n  \tif set_payload:\n  \tif slave:\n  \tif get_analog_in(3)/0.5952007 < 1.42:\n  \tzslam=0\n  \telse:\n  \tzslam=payload\n  \tend\n  \telse:\n  \tif get_digital_in(8) == False:\n  \tzmasm=0\n  \telse:\n  \tzmasm=payload\n  \tend\n  \tend\n  \tzload=zmasm+zslam+zsysm\n  \tset_payload(zload,[(zsysx*zsysm+zmasx*zmasm+zslax*zslam)/zload,(zsysy*zsysm+zmasy*zmasm+zslay*zslam)/zload,(zsysz*zsysm+zmasz*zmasm+zslaz*zslam)/zload])\n  \tend\n  \tmaster_grip_detected=False\n  \tmaster_lost_grip=False\n  \tslave_grip_detected=False\n  \tslave_lost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tmaster_grip_detected=True\n  \tend\n  \tif get_analog_in(3)/0.5952007>1.97:\n  \t\tslave_grip_detected=True\n  \tend\n  \tgrip_detected=False\n  \tlost_grip=False\n  \tif True == get_digital_in(8):\n  \t\tgrip_detected=True\n  \tend\n  \tzscale = (get_analog_in(2)-0.026)/2.9760034\n  \tzangle = zscale*1.57079633+-0.08726646\n  \tzwidth = 5.0+110*sin(zangle)\n  \tglobal measure_width = (floor(zwidth*10))/10-9.2\n  \tif slave:\n  \tslave_measure_width=measure_width\n  \telse:\n  \tmaster_measure_width=measure_width\n  \tend\n  \treturn grip_detected\n  end\n  set_tool_voltage(24)\n  set_tcp(p[0,0,0.18659,0,-0,-3.14159])\n{}{}'.format(
        cmd, "end\n"
    )
    strL = bytes(strL, encoding="utf8")
    socketConn.send(strL)
    wait_UR_state()
    time.sleep(2)


In [20]:
for i in range(0, 5):
    backward()
    right()
    forward()
    left()

回傳TCP POSE

In [ ]:
import socket
import struct

# 設定 UR5 機械手臂的 IP
UR5_IP = "192.168.50.155"  # 請根據你的 UR5 設定
PORT = 30003  # 30003 端口可回傳數據

def get_tcp_pose():
    """ 連接 UR5 並獲取當前 TCP 位姿 """
    try:
        # 建立 Socket 連線
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.connect((UR5_IP, PORT))

        # 接收數據
        data = s.recv(1108)  # 30003端口傳輸二進制數據，1108 bytes 是完整數據
        s.close()

        # 解析二進制數據
        tcp_pose = struct.unpack('!6d', data[444:492])  # TCP 位姿從第444到492 byte

        # 格式化輸出
        pose = {
            "X": tcp_pose[0],  # 米 (m)
            "Y": tcp_pose[1],
            "Z": tcp_pose[2],
            "RX": tcp_pose[3],  # 旋轉向量 (rad)
            "RY": tcp_pose[4],
            "RZ": tcp_pose[5]
        }
        return pose

    except Exception as e:
        print("錯誤:", e)
        return None

# 取得 TCP 位置
tcp_position = get_tcp_pose()
if tcp_position:
    print("當前 TCP 位姿：", tcp_position)

當前 TCP 位姿： (0.27911557166031903, 0.04595781183830465, 0.2182780639420304, 2.2165750198227214, -2.213687090584545, -0.00714169437140235)


關節角度

In [14]:
import socket
import struct

UR5_IP = "192.168.50.155"  # 機械手臂 IP
PORT = 30003  # 30003 端口可回傳狀態數據

def get_current_joint_positions():
    """ 取得 UR5 當前的關節角度 (qnear) """
    try:
        # 連線到 UR5
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.connect((UR5_IP, PORT))

        # 接收二進制數據
        data = s.recv(1108)
        s.close()

        # 解析關節角度 (第 252-300 Bytes, 6 個 double)
        joint_positions = struct.unpack('!6d', data[252:300])

        return list(joint_positions)  # 轉換成 Python List 方便使用

    except Exception as e:
        print("錯誤:", e)
        return None

# 取得 qnear
qnear = get_current_joint_positions()
if qnear:
    print("當前關節角度 (qnear):", qnear)


當前關節角度 (qnear): [0.559369683265686, -1.021700684224264, -2.0280678907977503, -1.6720245520221155, 1.5753684043884277, -2.583463493977682]


In [63]:
import socket
import struct
from datetime import datetime

# 設定 UR5 機械手臂的 IP
UR5_IP = "192.168.50.155"  # 請根據你的 UR5 設定
PORT = 30003  # 30003 端口可回傳數據
    
def get_current_tcp_joint_positions():
    """ 取得 UR5 當前的關節角度 (qnear) """
    try:
        # 連線到 UR5
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.connect((UR5_IP, PORT))

        # 接收二進制數據
        data = s.recv(1108)
        s.close()

        tcp_pose = struct.unpack('!6d', data[444:492])  # TCP 位姿從第444到492 byte
        joint_positions = struct.unpack('!6d', data[252:300]) # 解析關節角度 (第 252-300 Bytes, 6 個 double)
        tcp_pose = list(tcp_pose)  
        joint_positions = list(joint_positions)

        cmd = f"movej(get_inverse_kin(p{tcp_pose}, qnear={joint_positions}), a=1.3962634015954636, v=1.0471975511965976)" 
        
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open("ur_command.txt", "a") as file:
            file.write(f"{current_time} - {cmd}\n")

        return tcp_pose, joint_positions, cmd

    except Exception as e:
        print("錯誤:", e)
        return None

# 取得 TCP 位置
tcp_pose, joint_positions, ur_cmd = get_current_tcp_joint_positions()
print("當前 TCP 位姿：", tcp_position)
print("當前關節角度 (qnear):", joint_positions)
print("URScript 指令：", ur_cmd)

當前 TCP 位姿： (0.27911557166031903, 0.04595781183830465, 0.2182780639420304, 2.2165750198227214, -2.213687090584545, -0.00714169437140235)
當前關節角度 (qnear): [-0.14910489717592412, -1.5335834662066858, -1.6766279379474085, -1.5107877890216272, 1.567640781402588, -3.292394463215963]
URScript 指令： movej(get_inverse_kin(p[0.44637657652644636, -0.1796936048728122, 0.21765171451381016, 2.2164432829898786, -2.2133512820338246, -0.006792832987007367], qnear=[-0.14910489717592412, -1.5335834662066858, -1.6766279379474085, -1.5107877890216272, 1.567640781402588, -3.292394463215963]), a=1.3962634015954636, v=1.0471975511965976)


移動手臂

In [65]:
import socket
import time

# UR 機械手臂的 IP 與埠
UR_IP = "192.168.50.155"  # 替換成你的 UR IP
UR_PORT = 30001          # UR 的主要埠

# 建立 socket 連線
def connect_to_ur():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect((UR_IP, UR_PORT))
    return sock

# 傳送 URScript 指令
def send_command(sock, command):
    sock.send((command + "\n").encode('utf-8'))
    print(f"Sent command: {command}")
    time.sleep(0.1)  # 短暫延遲，確保指令送達

# 主程式
if __name__ == "__main__":
    try:
        # 連接到 UR
        ur_socket = connect_to_ur()
        print("Connected to UR successfully!")

        # 範例 URScript 指令：移動到指定位置
        move_command = 'movej(get_inverse_kin(p[0.44637657652644636, -0.1796936048728122, 0.21765171451381016, 2.2164432829898786, -2.2133512820338246, -0.006792832987007367], qnear=[-0.14910489717592412, -1.5335834662066858, -1.6766279379474085, -1.5107877890216272, 1.567640781402588, -3.292394463215963]), a=1.3962634015954636, v=1.0471975511965976)'
        send_command(ur_socket, move_command)

        # 等待幾秒觀察結果
        time.sleep(5)

    except Exception as e:
        print(f"Error: {e}")
    finally:
        ur_socket.close()
        print("Connection closed.")

Connected to UR successfully!
Sent command: movej(get_inverse_kin(p[0.44637657652644636, -0.1796936048728122, 0.21765171451381016, 2.2164432829898786, -2.2133512820338246, -0.006792832987007367], qnear=[-0.14910489717592412, -1.5335834662066858, -1.6766279379474085, -1.5107877890216272, 1.567640781402588, -3.292394463215963]), a=1.3962634015954636, v=1.0471975511965976)
Connection closed.
